Groundwater | Transport Modeling Track

# Step 2: Perceptual Model – From Flow to Transport

> **The 10 steps at a glance:** 1-Goal → **2-Perceptual** → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 1 you set the objective: a **contaminant spills upgradient** of a **groundwater heat-exchange (GWHE) doublet**, and we ask whether the plume reaches the doublet's **extraction well** — which doubles as the monitoring / compliance well — above the regulatory threshold (and when), or **bypasses** it laterally. The doublet runs **flow-only** (clean injection +Q, extraction −Q); its forced-gradient flow field is what captures or bypasses the spill. Now we build a **perceptual model for transport** – what additional understanding do we need beyond the calibrated flow model?

## Timing and Learning Objectives

| | |
|---|---|
| **Time** | ~30–35 min core + ~10 min optional sections |
| **Prerequisites** | Flow track Steps 1–2, Transport track Step 1 |

**Learning objectives:**

By the end of this notebook you will be able to:

1. Calculate **groundwater (seepage) velocity** from Darcy flux and effective porosity
2. Estimate **hydrodynamic dispersion** and set the longitudinal/transverse dispersivities $\alpha_L,\ \alpha_T$
3. Assess the **grid Peclet number** and the TVD advection scheme for the Limmat Valley
4. Explain how the **spill source** is represented (a zero-water mass-loading source) and why it leaves the flow field undisturbed
5. Describe the **minimal reaction** a non-conservative contaminant needs — sorption (retardation) or first-order decay — and how each reshapes the breakthrough
6. Identify the main **simplifications** in the perceptual model — including the transverse-smearing limitation

> **You will MODIFY a provided model, not build one.** The GWT (groundwater transport) model is supplied in Step 4; here we establish the parameters and concepts behind it.

In [ ]:
# Setup
import sys
import os

import pyproj
os.environ['PROJ_LIB'] = pyproj.datadir.get_data_dir()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file, get_default_data_folder
from map_utils import display_concessions_map
from shared_functions import check_task_with_solution, create_multiple_choice

DATA_DIR = get_default_data_folder()

---
## 1 — From Flow to Transport

The calibrated flow model gives us the **velocity field** (specific discharge $q$). To simulate transport of the contaminant we need additional information:

| Flow model gave us | Transport adds |
|---|---|
| Hydraulic conductivity $K$ | Effective porosity $n_e$ → **velocity** $v = q / n_e$ |
| Hydraulic gradient $i$ | Dispersivity $\alpha_L,\ \alpha_T$ → **hydrodynamic dispersion** |
| Well pumping / injection rates (flow only) | Spill location & concentration → **contaminant source** (adds no water) |
| Water balance (fluxes) | Concentration of each flux → **transport boundary conditions** |

This notebook works through each of these additions.

### Where do these numbers come from?

Transport parameters differ a lot in how well we can actually pin them down:

| Parameter | How it is obtained | In this course |
|---|---|---|
| Hydraulic conductivity $K$, gradient $i$ | Calibrated flow model (flow track) | inherited |
| Effective porosity $n_e$ | Rarely measured at model scale — literature by material (gravel ≈ 0.15–0.30; Freeze & Cherry 1979) and/or **calibrated** against tracer or groundwater-age data | literature default **0.20** |
| Seepage velocity $v$ | **Derived, not measured**: $v = q/n_e$ from the flow field and porosity. Measurable *locally* by borehole/point-dilution, tracer, or heat-tracer tests | derived |
| Dispersivity $\alpha_L,\ \alpha_T$ | Field tracer tests; strongly **scale-dependent** (Gelhar et al. 1992) | literature defaults **10 / 1 m** |

> **Is effective porosity ever measured directly?** On small cores, yes (column tracer tests, NMR) — but small-sample values miss field-scale pore connectivity, so the transport-relevant *effective (kinematic)* porosity at model scale is essentially a **literature-guided, calibrated** parameter, not a surveyed one. Groundwater **velocity** is likewise *derived* from the flow solution rather than measured across a basin; a local tracer or borehole-dilution test can pin it along a single path. Both carry a mild **scale dependence** — smaller than dispersivity's, but real.

---
## 2 — Groundwater Velocity

The **specific discharge** (Darcy flux) $q = Ki$ tells us the volume of water crossing a unit area per unit time, but it is not the speed at which water (or a tracer) actually moves through the pore space.

The **seepage velocity** (average linear velocity) is:

$$v = \frac{q}{n_e} = \frac{Ki}{n_e}$$

For the Limmat Valley glaciofluvial gravels, effective porosity $n_e$ is typically **0.15–0.30** (Freeze & Cherry 1979, Table 2.4). We adopt $n_e = 0.20$ as a reasonable central estimate for well-sorted gravel.

> **Predict before you run:** Near the doublet's extraction well, pumping steepens the local hydraulic gradient. Before running the next cell, predict: will near-well seepage velocities be **higher or lower** than the domain-average velocity? Note your reasoning, then check it against the printed domain average.

In [ ]:
# Representative values from the calibrated flow model
K = 2.31e-3        # hydraulic conductivity [m/s]  (≈ 200 m/d, calibrated value)
dh = 18.0          # head difference across model domain [m]
dx = 7900.0        # domain extent in flow direction [m]
i = dh / dx        # hydraulic gradient [-]
n_e = 0.20         # effective porosity [-]

# Specific discharge
q = K * i  # [m/s]

# Seepage velocity
v = q / n_e  # [m/s]
v_m_per_day = v * 86400  # convert to m/day

print(f"Hydraulic gradient:     i = {i:.4f}")
print(f"Specific discharge:     q = {q:.2e} m/s  ({q * 86400:.2f} m/d)")
print(f"Seepage velocity:       v = {v:.2e} m/s  ({v_m_per_day:.1f} m/d)")

# Travel time across ~4.5 km domain
L_domain = 4500  # [m]
travel_time_days = L_domain / v_m_per_day
print(f"\nTravel time across {L_domain/1000:.1f} km domain: ~{travel_time_days:.0f} days ({travel_time_days/365:.0f} years)")
print(f"\nNote: this is a domain-average velocity. Near a doublet's extraction")
print(f"well, pumping-induced gradients create local velocities well above it.")

**Key insight:** The seepage velocity (~2.3 m/day on average) is faster than the specific discharge (~0.45 m/day) by a factor of $1/n_e = 5$. This distinction matters for:

- **Wellhead / capture-zone protection** — defined by travel time of water (velocity), not flux
- **Contaminant arrival times** — near a pumping well, pumping-induced gradients mean a contaminant spilled a few hundred metres upgradient could arrive within weeks to months
- **Capture vs bypass** — whether an upgradient spill is drawn into the extraction well or swept past depends directly on these local velocities and the well's capture zone

### Checkpoint 1 — Seepage Velocity

In [ ]:
check_task_with_solution('task_t02_checkpoint_1')

---
## 3 — Dispersion

In a porous medium, a solute front does not advance as a sharp plug — it **spreads** due to:

1. **Mechanical dispersion** — velocity variations at the pore scale cause differential advection
2. **Molecular diffusion** — random motion of dissolved molecules (usually negligible in fast-flowing gravels)

The **longitudinal hydrodynamic dispersion coefficient** combines both:

$$D_L = \alpha_L \cdot v + D_m^*$$

where $\alpha_L$ is the longitudinal dispersivity [m] and $D_m^*$ is the effective molecular diffusion coefficient. In porous media $D_m^* \approx 10^{-9}$ $m^2/s$, which in the model's time units (days) is $D_m^* \approx 8.64\times10^{-5}$ $m^2/d$ (the $\times 86400$ s/d conversion — never use the unconverted $10^{-9}$ in a days-based model).

### Scale-dependent dispersivity

Dispersivity is notoriously **scale-dependent**: it increases with the distance over which transport is observed (Gelhar et al. 1992). For our model scale (~1–5 km):

| Observation scale | Typical $\alpha_L$ range | Reference |
|---|---|---|
| Laboratory column (0.1–1 m) | 0.001–0.01 m | Freeze & Cherry 1979 |
| Field tracer test (10–100 m) | 0.1–10 m | Gelhar et al. 1992 |
| Regional model (1–10 km) | 5–100 m | Gelhar et al. 1992 |

For the Limmat Valley at our model scale we adopt **$\alpha_L = 10$ m**. The transverse dispersivity is taken as one-tenth of the longitudinal value, **$\alpha_T = \alpha_L / 10 = 1$ m** — a tracer spreads roughly ten times less across the flow direction than along it.

> **Forward pointer:** $\alpha_T = 1$ m is small. In the grid-Peclet check below you will see that resolving *transverse* spreading is far harder than longitudinal spreading — the **transverse Peclet number stays well above the accuracy target ($Pe_{grid}\lesssim 2$) even on a refined grid**. Step 4 returns to this as the *transverse-smearing* limitation.

In [ ]:
# Dispersion parameters
alpha_L = 10.0          # longitudinal dispersivity [m]
alpha_T = alpha_L / 10  # transverse dispersivity [m]  → 1 m
D_m_star = 1e-9         # effective molecular diffusion [m²/s]  (= 8.64e-5 m²/d in model days-units)

D_L = alpha_L * v + D_m_star  # longitudinal dispersion coefficient [m²/s]
D_T = alpha_T * v + D_m_star  # transverse  dispersion coefficient [m²/s]

print(f"Longitudinal dispersivity:  alpha_L = {alpha_L} m")
print(f"Transverse dispersivity:    alpha_T = {alpha_T} m  (= alpha_L / 10)")
print(f"Dispersion coefficient:     D_L = {D_L:.2e} m²/s")
print(f"  of which mechanical:      alpha_L * v = {alpha_L * v:.2e} m²/s")
print(f"  of which diffusion:       D_m* = {D_m_star:.0e} m²/s  (= 8.64e-5 m²/d)")
print(f"\n→ Mechanical dispersion dominates over molecular diffusion by a")
print(f"  factor of {alpha_L * v / D_m_star:.0e}. Molecular diffusion is negligible.")

# --- Grid Peclet number: numerical resolution criterion (Pe_grid ≲ 2) ---
# Note: v is in m/s and dx in m, so Pe is dimensionless regardless of time units;
#       cp1 reports v in m/d, here we keep SI (m/s) — the Peclet ratio is unaffected.
dx_grid = 49  # typical cell size in the provided DISV grid [m] (model median)
Pe_L = v * dx_grid / D_L   # longitudinal grid Peclet
Pe_T = v * dx_grid / D_T   # transverse  grid Peclet

print(f"\nNative grid (Δx = {dx_grid} m):")
print(f"  Longitudinal grid Peclet:  Pe_L = {Pe_L:.1f}   (target ≤ 2)")
print(f"  Transverse grid Peclet:    Pe_T = {Pe_T:.0f}   (target ≤ 2 — badly under-resolved)")
print(f"  Critical cell size for Pe_L ≤ 2:  Δx ≤ {2 * D_L / v:.0f} m = 2 × alpha_L")
print(f"  Critical cell size for Pe_T ≤ 2:  Δx ≤ {2 * D_T / v:.0f} m = 2 × alpha_T")

Our **longitudinal** grid Peclet number is ~5, exceeding the classical limit of 2. With a **central-difference** advection scheme this would cause spurious oscillations. MODFLOW 6 GWT supports **TVD (Total Variation Diminishing)** advection, which stays stable at higher grid Peclet numbers, so the longitudinal front is usable on the native grid (at the cost of some numerical dispersion).

The **transverse** grid Peclet number is far worse (~49 on the native grid, and still ~10 even on the refined grid used in Step 4). TVD keeps the solution stable, but the transverse plume is narrower than the grid can resolve, so **lateral concentration contours are under-resolved and not defensible**. We flag this now and return to it as a judgment limitation in Step 4 — it is *taught*, not *fixed*.

<details>
<summary><b>Optional: Grid Peclet criterion — when does it matter?</b> (click to expand)</summary>

The **grid Peclet number** is a numerical (not physical) criterion:

$$Pe_{grid} = \frac{v \cdot \Delta x}{D} \lesssim 2$$

Longitudinally this simplifies to $\Delta x \lesssim 2\,\alpha_L$ (= 20 m for $\alpha_L = 10$ m); transversely to $\Delta x \lesssim 2\,\alpha_T$ (= 2 m for $\alpha_T = 1$ m). Resolving transverse spreading therefore demands a ~10× finer grid than longitudinal spreading — rarely affordable over a km-scale domain.

**Why we rely on TVD instead of refining everywhere:**
- TVD (upstream-weighted) schemes are stable regardless of $Pe_{grid}$ — they handle sharp fronts without oscillation
- The tradeoff is **numerical dispersion**, which adds to the physical dispersion
- Refining the entire grid to a few metres would multiply cell count and runtime prohibitively

**When local refinement IS used:** around the doublet, where concentrations are highest and gradients steepest, local refinement (Step 4) reduces $Pe_L$ to ~1 — but $Pe_T$ still stays near 10, which is exactly the transverse-smearing limitation above.

</details>

### Checkpoint — Grid Peclet Number

In [ ]:
check_task_with_solution('task_t02_checkpoint_pe')

---
### Reactive solutes — sorption and decay

We **start from a conservative tracer** ($R = 1$: it moves with the water, neither retained nor degraded) because that isolates the transport the flow field controls. But **5 of the 9 case-study contaminants are reactive**, so a real breakthrough usually needs one incremental process on top — and no case study needs both at once.

**1. Sorption → retardation.**   
A solute that partitions onto the aquifer grains travels *slower* than the water. For linear equilibrium sorption the **retardation factor** is

$$R = 1 + \frac{\rho_b \, K_d}{n_e}$$

with dry bulk density $\rho_b$ [kg/m³], distribution coefficient $K_d$ [m³/kg], and effective porosity $n_e$. The sorbing front advances at $v/R$, so arrival is delayed by the factor $R$. Across the assigned contaminants ($\rho_b = 1800$ kg/m³, $n_e = 0.20$):

| Contaminant (group) | $K_d$ | $R = 1 + \rho_b K_d / n_e$ | Behaviour |
|---|---|---|---|
| PCE (6) | 0.08 mL/g | ≈ 1.7 | slight retardation |
| Atrazine (8) | 0.30 mL/g | ≈ 3.7 | moderate |
| Chromium (4) | 2.0 mL/g | ≈ 19 | strong |

**2. First-order decay.**   
A solute that biodegrades or transforms loses dissolved mass at a rate proportional to its concentration: $\partial C/\partial t = -\lambda C$, with decay constant $\lambda$ [1/T] and half-life $t_{1/2} = \ln 2 / \lambda$. Decay **lowers the peak and the plateau** (for a continuous source, the steady concentration at the well) but — unlike sorption — does **not** delay the front. Two of the case-study contaminants decay: **benzene** (group 2, aerobic biodegradation, $t_{1/2}\approx139$ d) and **ammonium** (group 7, nitrification, $t_{1/2}\approx35$ d).

Both processes — sorption (1) and first-order decay (2) — are configured in the same MODFLOW 6 GWT package, **MST** (Mass Storage and Transfer): `sorption`, `bulk_density`, `distcoef` for retardation; `first_order_decay`, `decay` for decay. Your group's reaction parameters are **pinned** in `case_config_transport.yaml`: the provided model reads them, you interpret the result — you do not retune them.

<details>
<summary><b>What about decay <i>products</i>? (click to expand)</b></summary>

Some contaminants don't simply disappear — they decay into **daughter products** that must themselves be tracked, and which can be *more* mobile or toxic than the parent (classic example: TCE → cis-DCE → vinyl chloride).

**Can MODFLOW 6 GWT do this?** Not directly. One GWT model simulates **one species**, and its first-order decay merely *removes* mass — no daughter is produced. Several species can run in one simulation, but they are **non-interacting**: MODFLOW 6 does not feed a parent's decay into a daughter. A genuine decay chain needs custom species coupling (via the MODFLOW API) or a dedicated reactive-transport code — **RT3D**, **MT3D-USGS** (RCT reaction package), or **PHT3D** (PHREEQC-coupled geochemistry). This course deliberately treats each contaminant as a **single species** — which is why TCE (group 0) is modeled as a conservative tracer rather than a TCE→DCE→VC chain.
</details>

<details>
<summary><b>Worked example — atrazine retardation (click to expand)</b></summary>

Atrazine (group 8): $K_d = 0.30$ mL/g $= 3\times10^{-4}$ m³/kg, $\rho_b = 1800$ kg/m³, $n_e = 0.20$:

$$R = 1 + \frac{1800 \times 3\times10^{-4}}{0.20} = 1 + \frac{0.54}{0.20} = 3.7$$

so atrazine's front travels about **3.7× slower** than a conservative tracer — a pulse a conservative tracer delivers in ~100 days arrives in ~370 days, correspondingly more spread out and diluted.
</details>

**Checkpoint — compute the retardation factor** for chromium (group 4, the strong-sorption case) in the cell that follows:

In [ ]:
check_task_with_solution('task_t02_checkpoint_2')

---
## 4 — GWHE Doublet Locations in the Limmat Valley

The Limmat Valley is one of Switzerland's most intensely used aquifers for **groundwater heat exchange (GWHE)** — **injection–extraction doublet wells** that pump groundwater, exchange heat for heating or cooling, and reinject it nearby. Although their purpose is thermal, each doublet is also a **hydraulic** feature: it withdraws water at one well and returns it at another, setting up a **forced-gradient flow field** with a capture zone around the extraction well — the field that decides whether an upgradient contaminant spill is drawn in or swept past.

In the regulatory data these appear as groundwater heat-pump (WPG = *Wärmepumpe Grundwasser*) and cooling-water (KW = *Kühlwasser*) concessions. Let's map them in our model area.

In [ ]:
# Download well and boundary data
wells_path = download_named_file(name='wells', data_type='gis')
model_boundary_path = download_named_file(name='model_boundary', data_type='gis')

# Read and clip to model boundary
gdf_boundary = gpd.read_file(model_boundary_path)
wells_gdf = gpd.read_file(wells_path, layer='GS_GRUNDWASSERFASSUNGEN_OGD_P')
wells_gdf = wells_gdf.clip(gdf_boundary)

# Add concession ID
wells_gdf['concession_id'] = wells_gdf['GWR_ID'].str.split('_').str[0]

# Filter active wells
is_decommissioned = wells_gdf['BESCHREIBUNG'].str.contains('aufgehoben', case=False, na=False)
is_unused = wells_gdf['BESCHREIBUNG'].str.contains('ungenutzt', case=False, na=False)
wells_active = wells_gdf[~is_decommissioned & ~is_unused].copy()

# Filter for GWHE doublet use: WPG (heat pump) or KW (cooling water)
is_gwhe = wells_active['NUTZART'].str.contains('WPG|KW', case=False, na=False)
gwhe_wells = wells_active[is_gwhe].copy()

n_gwhe_concessions = gwhe_wells['concession_id'].nunique()
n_all_concessions = wells_active['concession_id'].nunique()

print(f"Active concessions in model area:  {n_all_concessions}")
print(f"GWHE doublet concessions (WPG/KW): {n_gwhe_concessions}")
print(f"Fraction GWHE:                     {n_gwhe_concessions/n_all_concessions:.0%}")

In [ ]:
# Map of GWHE doublet concessions
display_concessions_map(
    gwhe_wells,
    boundary_gdf=gdf_boundary,
    map_title="GWHE Doublet Concessions (WPG/KW) — Limmat Valley (colour = concession; shape = use type — see legend)"
)

**Observations from the map:**

- GWHE doublets are **concentrated in the city centre** (Zurich districts 1–5), reflecting high heating/cooling demand
- Individual doublets are not represented in our base model — students activate a specific doublet, and place an upgradient spill, in the application step
- Each doublet is a candidate **capture-vs-bypass** setting: whether a contaminant spilled upgradient reaches the extraction (monitoring) well runs through exactly the velocity and dispersion field we just characterised. *(The doublet's own thermal concern — recirculation of reinjected water back to the extraction well — is the same flow field seen from the operator's side; Step 8 revisits it.)*

### Representing the spill source (preview)

How does the model put contaminant *into* the aquifer at the spill? In Step 4 the spill is a **zero-water mass-loading source** (the MODFLOW 6 **SRC** package): you specify the **released mass** — a loading rate [M/T] over the release period — at a small set of cells at the spill location. Crucially it **adds no water**, so the doublet's flow field (clean injection $+Q$, extraction $-Q$) is undisturbed and the question stays purely *does the existing flow field carry the plume to the well?* And because you set the *mass* directly, the source is **grid-independent** — unlike a fixed-concentration cell, whose injected mass floats with the grid. For a **pulse** spill the loading is on during the release period and then off, letting the plume migrate (and react) freely; for a **continuous** source it stays on.

> **Contrast — CNC and water-carrying sources.** A **CNC (constant-concentration)** cell instead *pins* the concentration and lets the model supply whatever mass holds it — grid- and flow-dependent — so it fits only a genuinely *held-concentration* source (a NAPL pool dissolving at solubility). And when the source *is* a well that injects **water** — e.g. the geothermal **recirculation** study in Step 8, where a tracer rides the reinjected water — mass enters as WEL flux × concentration through **SSM**. Our upgradient spill carries no water and has a known mass, so **SRC** is the natural representation. (Full build: Step 4.)

### Checkpoint — GWHE Doublet Distribution

In [ ]:
create_multiple_choice('task_t02_checkpoint_3')

---
## 5 — Could We Be Wrong?

Every perceptual model involves simplifications. Here are the main **conceptual alternatives** for our contaminant-transport model:

| Assumption | Alternative | Impact if wrong |
|---|---|---|
| Source term known (released **mass / loading rate**, location, release duration) | The true **spill mass and duration** are usually unknown | Peak concentration scales with the assumed loading (arrival timing is set by advection, dispersion, and retardation — not by loading) |
| Contaminant enters the saturated zone **directly** | Real spills start at the ground surface or a leaking sewer and first migrate through the **unsaturated (vadose) zone** | The vadose zone delays, attenuates, and smears the input — our model starts the clock at the water table |
| Only the pinned reaction (sorption **or** decay) | Coupled sorption+decay, non-linear sorption, or extra reactions | Real attenuation differs — e.g. benzene and ammonium in reality *both* sorb and degrade; here each group's single reaction is pinned, not fitted |
| Uniform $\alpha_L = 10$ m | Larger/smaller or spatially variable $\alpha_L$ | More/less front spreading; peak concentration and arrival shift |
| $\alpha_T = \alpha_L/10 = 1$ m, **adequately resolved** | Transverse spreading **under-resolved on any feasible grid** ($Pe_T \approx 49$ native, ~10 refined) | **The transverse plume is sub-cell (narrower than the grid), so lateral concentration contours / *area* claims are not defensible — the honest lateral answer is particle tracking (08t)** |
| Single vertical layer | Multiple layers to resolve vertical structure | Vertical concentration gradients missed |
| Steady-state flow field | Transient flow (seasonal water-table changes) | Seasonal changes in path and travel time missed |
| Uniform porosity $n_e = 0.20$ | Spatially variable $n_e$ (0.15–0.30) | Local velocity variations, different travel times |
| TVD advection adequate on native grid | Numerical dispersion biases the front | Front artificially smoothed; local refinement (Step 4) needed near the doublet |

The flagship limitation is the **transverse-resolution** one: even after local refinement the transverse plume is narrower than the cells (the transverse grid Peclet stays well above 2), so the model's **lateral** plume width is under-resolved and lateral concentration contours are not defensible. (A high $Pe_T$ by itself does *not* prove *numerical* smearing — on flow aligned with the grid the TVD scheme adds little; the lateral limitation here is the sub-cell transverse plume plus the doublet's convergent, grid-oblique flow, which does add some cross-wind numerical dispersion.) The defensible lateral answer is **particle tracking** (08t Rung D), not the smeared concentration field. This is *taught*, not *fixed* — it shapes what we can defend in the application step (receptor arrival and well concentration: yes; contaminated *area* contours: no). Step 4 makes this concrete.

Two more are worth flagging for a *spill* scenario specifically: we rarely know the true **source strength and duration**, and we start the contaminant **at the water table** — a real surface or sewer spill must first travel down through the **unsaturated (vadose) zone**, which delays and smooths the loading before aquifer transport even begins.

*Before moving on, take a moment to reflect:*

> **Which assumption above do you think is most consequential for the capture-vs-bypass question? Why?**
>
> Consider both (a) how likely the assumption is to be wrong and (b) how much impact it would have on the predicted plume arrival at the extraction (monitoring) well.

---
## What You're Taking Forward

| Parameter | Value | Unit | Source |
|---|---|---|---|
| Effective porosity $n_e$ | 0.20 | — | Literature (glaciofluvial gravel) |
| Dry bulk density $\rho_b$ | 1800 | kg/m³ | Literature (gravel); used in retardation |
| Seepage velocity $v$ | ~2.3 (domain average) | m/day | $v = Ki/n_e$ from calibrated model |
| Longitudinal dispersivity $\alpha_L$ | 10 | m | Gelhar et al. 1992 (model scale) |
| Transverse dispersivity $\alpha_T$ | 1 | m | $\alpha_T = \alpha_L/10$ |
| Molecular diffusion $D_m^*$ | $8.64\times10^{-5}$ | m²/d | $=10^{-9}$ m²/s; negligible vs. mechanical dispersion |
| Native longitudinal grid Peclet $Pe_L$ | ~5 | — | $v\,\Delta x / D_L$ (> 2 → use TVD) |
| Native transverse grid Peclet $Pe_T$ | ~49 | — | $v\,\Delta x / D_T$ (under-resolved; ~10 even refined) |
| Advection scheme | TVD | — | Stable for $Pe_{grid} > 2$ |
| Spill source | zero-water SRC mass loading | — | released mass set directly; grid-independent; adds no water |
| Reaction (if non-conservative) | $R = 1 + \rho_b K_d/n_e$ or first-order decay $\lambda$ | — | pinned per group; MST package |

> **Next step:** In [Step 3 (MODFLOW for Transport)](./03t_modflow_transport.ipynb), we translate this perceptual model into a MODFLOW 6 **GWT (Groundwater Transport)** model configuration.

---
## References

- Doppler, T. et al. (2007). Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. *Journal of Hydrology*, 347(1–2), 177–187. [doi:10.1016/j.jhydrol.2007.09.017](https://doi.org/10.1016/j.jhydrol.2007.09.017)
- Fetter, C.W. (2001). *Applied Hydrogeology*. 4th ed. Pearson (Prentice Hall). ISBN 978-1-292-02290-1 (print); e-ISBN 978-1-292-03608-3.
- Freeze, R.A. & Cherry, J.A. (1979). *Groundwater*. Prentice Hall. [free (Groundwater Project)](https://gw-project.org/books/groundwater/)
- Gelhar, L.W., Welty, C. & Rehfeldt, K.R. (1992). A critical review of data on field-scale dispersion in aquifers. *Water Resources Research*, 28(7), 1955–1974. [doi:10.1029/92WR00607](https://doi.org/10.1029/92WR00607)
- Langevin, C.D., Provost, A.M., Panday, S. & Hughes, J.D. (2022). *Documentation for the MODFLOW 6 Groundwater Transport Model*. USGS Techniques and Methods 6-A61. [doi:10.3133/tm6A61](https://doi.org/10.3133/tm6A61)
- Zheng, C. & Bennett, G.D. (2002). *Applied Contaminant Transport Modeling*. 2nd ed. Wiley. [ISBN 978-0471384779](https://www.wiley.com/en-us/Applied+Contaminant+Transport+Modeling%2C+2nd+Edition-p-9780471384779)
- Canton of Zurich (2023). *Gewässer- und Grundwasser-Jahrbuch*. AWEL. [zh.ch](https://www.zh.ch/de/umwelt-tiere/wasser-gewaesser/gewaesserschutz/gewaesserqualitaet.html)